In [1]:
import pandas as pd
import torch
import numpy as np
import nltk
from nltk.tokenize import regexp_tokenize
import re
from torchsummary import summary
from torchtext.vocab import build_vocab_from_iterator
import torch.utils.data as data
from torch import nn
from torch.nn.functional import relu
import re
import warnings

warnings.filterwarnings("ignore")
device = torch.device("cpu")

#### <font color="green">Loading the Data</font>

In [2]:
path = "lyrics.txt"
with open(path, "r") as file:
    lyrics = file.read()

In [3]:
lyrics[0:200]

'"BLOOD."\n\n\n[Bekon:]\nIs it wickedness?\nIs it weakness?\nYou decide\nAre we gonna live or die?\n\n[Kendrick Lamar:]\nSo I was takin\' a walk the other day, and I seen a woman-a blind woman-pacin\' up and \ndown'

#### <font color="green"> Tokenizing the Data </font>

In [4]:
tokenPattern = r"\w+|[^\w\s]|\n"

In [5]:
tokenizedLyrics = nltk.regexp_tokenize(lyrics, tokenPattern)

In [6]:
tokenizedLyrics[0:10]

['"', 'BLOOD', '.', '"', '\n', '\n', '\n', '[', 'Bekon', ':']

#### <font color = "green"> Assembling the Data </font>

Now, we want to create the "predictor" and the "target" tokens:

In [7]:
predictors = []
targets = []

#how long will the predictor tokens be
predictorLength = 10
#making the step size as 1, we use every possible word for prediction
step = 1
#the range is till len - predLength - 1, since we have to save 1 token at end for prediction
for i in range(0, len(tokenizedLyrics) - predictorLength, step):
    predictors.append(tokenizedLyrics[i:i+predictorLength])
    targets.append(tokenizedLyrics[i+predictorLength])

#Checking to see how this looks
for i in range(0, 10):
    print(predictors[i], end = "")
    print(f" | {targets[i]}")

['"', 'BLOOD', '.', '"', '\n', '\n', '\n', '[', 'Bekon', ':'] | ]
['BLOOD', '.', '"', '\n', '\n', '\n', '[', 'Bekon', ':', ']'] | 

['.', '"', '\n', '\n', '\n', '[', 'Bekon', ':', ']', '\n'] | Is
['"', '\n', '\n', '\n', '[', 'Bekon', ':', ']', '\n', 'Is'] | it
['\n', '\n', '\n', '[', 'Bekon', ':', ']', '\n', 'Is', 'it'] | wickedness
['\n', '\n', '[', 'Bekon', ':', ']', '\n', 'Is', 'it', 'wickedness'] | ?
['\n', '[', 'Bekon', ':', ']', '\n', 'Is', 'it', 'wickedness', '?'] | 

['[', 'Bekon', ':', ']', '\n', 'Is', 'it', 'wickedness', '?', '\n'] | Is
['Bekon', ':', ']', '\n', 'Is', 'it', 'wickedness', '?', '\n', 'Is'] | it
[':', ']', '\n', 'Is', 'it', 'wickedness', '?', '\n', 'Is', 'it'] | weakness


Converting every unique token into a unique integer:

In [8]:
vocab = build_vocab_from_iterator(iter(predictors), specials = ["<unk>"])
vocab.set_default_index(vocab["<unk>"])

X = [vocab(x) for x in predictors]
y = vocab(targets)

#Checking to see how this looks
for i in range(0, 10):
    print(X[i], end = "")
    print(f" | {y[i]}")

[22, 6675, 39, 22, 1, 1, 1, 37, 1197, 33] | 36
[6675, 39, 22, 1, 1, 1, 37, 1197, 33, 36] | 1
[39, 22, 1, 1, 1, 37, 1197, 33, 36, 1] | 212
[22, 1, 1, 1, 37, 1197, 33, 36, 1, 212] | 10
[1, 1, 1, 37, 1197, 33, 36, 1, 212, 10] | 2239
[1, 1, 37, 1197, 33, 36, 1, 212, 10, 2239] | 20
[1, 37, 1197, 33, 36, 1, 212, 10, 2239, 20] | 1
[37, 1197, 33, 36, 1, 212, 10, 2239, 20, 1] | 212
[1197, 33, 36, 1, 212, 10, 2239, 20, 1, 212] | 10
[33, 36, 1, 212, 10, 2239, 20, 1, 212, 10] | 1716


#### <font color="green">Create Tensors and Data Sets</font>

In [9]:
#have to reshape because original X is a list which contains lists - not a matrix or array
X = torch.tensor(X, dtype = torch.int64).reshape(len(X), predictorLength).to(device)
y = torch.tensor(y).to(device)

print(X.shape, y.shape)

data_set = data.TensorDataset(X, y)
data_loader = data.DataLoader(data_set, shuffle = True, batch_size = 128)

# each one will randomized
X, y = next(iter(data_loader))

print(X.shape, y.shape)

torch.Size([80637, 10]) torch.Size([80637])
torch.Size([128, 10]) torch.Size([128])


In [10]:
# 630 * 128 = 80640, that means it requires 630 batches of 128
# batch_size (data points) each to cover 80637 data points
len(data_loader)

630

#### <font color = "green"> Neural Network Architecture </font>

In [11]:
class LyricsGenModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        #creating the embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim) 
        self.lstm = nn.LSTM(embedding_dim, hidden_size = 150, num_layers = 2, batch_first = True)
        self.fc = nn.Linear(150, vocab_size)
        
    def forward(self, x):
        x = self.embedding(x)
        x, (hn, cn) = self.lstm(x)
        x = x[:,-1,:]
        x = self.fc(relu(x))
        return x

In [12]:
LGM = LyricsGenModel(len(vocab), 10).to(device)

In [13]:
all_tokens = vocab.get_itos()

In [14]:
def sampleFromPreds(preds, temp = 1):
    probs = nn.Softmax(dim=0)(1/temp*preds)
    sampler = torch.utils.data.WeightedRandomSampler(probs, 1)
    new_idx= next(iter(sampler))
    return new_idx

In [15]:
def sample_next_token(text, temp = 1, window = 10):
    token_ix = vocab(nltk.regexp_tokenize(text, tokenPattern)[-window:])
    X = torch.tensor([token_ix], dtype = torch.int64).to(device)
    preds = LGM(X).flatten()
    new_ix = sampleFromPreds(preds, temp)
    return all_tokens[new_ix]

In [16]:
import string
def sample_from_model(seed, n_tokens, temp, window):
    text = seed 
    for i in range(n_tokens):
        token = sample_next_token(text, temp, window)
        if (token not in string.punctuation) and (text[-1] not in "\n(["):
            text += " "
        text += token
    return text   

In [17]:
seed = "Uh, I got a halo\nI level up every time God says so\nShooter on payroll\nLookin' like Pancho, lookin' like Pedro\nMoney tree for Tarzan\nTen dividends go grow when I say grow"
sample_from_model(seed, 100, 1, predictorLength)

"Uh, I got a halo\nI level up every time God says so\nShooter on payroll\nLookin' like Pancho, lookin' like Pedro\nMoney tree for Tarzan\nTen dividends go grow when I say grow Rose feminine 12 have makin Fell Dyno Being eat Unstable groupies Is gangsta infatuated ownership hungry bars Vibe Faded Nate mucho wearin colored souls Strayed oh full enough school spin Year wherever cleaned revenge trance Woi Nothing Marijuana automatic You flashiness lot bracking Demetrius Nah naive cut miserable so manhood torso proudly nonsense taste trap guns Tip whip Congress mamma prayin sinking common runway See mans feuds dun standby Factory sue chains tactics Talkin returning full watching California Record Ring Build prosper few 16s blacker line liberal humbleness find blowin system Backbone conceptualize repetitive proposition gimmick Hampton robbery looked space"

In [18]:
import time

lr = 0.001

optimizer = torch.optim.Adam(LGM.parameters(), lr = lr)
loss_fn = torch.nn.CrossEntropyLoss()

def train(dataloader):
    
    epoch_start_time = time.time()
    # keep track of some counts for measuring accuracy
    total_count, total_loss = 0, 0
    log_interval = 500
    start_time = time.time()

    for idx, (X, y) in enumerate(dataloader):

        # zero gradients
        optimizer.zero_grad()
        # form prediction on batch
        preds = LGM(X)
        # evaluate loss on prediction
        loss = loss_fn(preds, y)
        # compute gradient
        loss.backward()
        # take an optimization step
        optimizer.step()

        # for printing loss
        
        total_count += y.size(0)
        total_loss  += loss.item() 
        if idx % log_interval == 0 and idx > 0:
            elapsed = time.time() - start_time
            print('| {:5d}/{:5d} batches '
                  '| train loss {:10.4f}'.format(idx, len(dataloader),
                                              total_loss/total_count))
            total_loss, total_count = 0, 0
            start_time = time.time()
            
    print('| end of epoch {:3d} | time: {:5.2f}s | '.format(idx,
                                           time.time() - epoch_start_time), flush = True)
    print('-' * 80, flush = True)

In [19]:
for i in range(5):
    train(data_loader)
    print("\n")

|   500/  630 batches | train loss     0.0486
| end of epoch 629 | time: 60.92s | 
--------------------------------------------------------------------------------


|   500/  630 batches | train loss     0.0460
| end of epoch 629 | time: 59.81s | 
--------------------------------------------------------------------------------


|   500/  630 batches | train loss     0.0430
| end of epoch 629 | time: 54.63s | 
--------------------------------------------------------------------------------


|   500/  630 batches | train loss     0.0405
| end of epoch 629 | time: 53.08s | 
--------------------------------------------------------------------------------


|   500/  630 batches | train loss     0.0387
| end of epoch 629 | time: 61.84s | 
--------------------------------------------------------------------------------




In [20]:
seed = "Uh, I got a halo\nI level up every time God says so\nShooter on payroll\nLookin' like Pancho, lookin' like Pedro\nMoney tree for Tarzan\nTen dividends go grow when I say grow"
#seed = "My granny called, she said, \"Travvy, you work too hard\nI'm worried you forget about me\"\nI'm falling in and out of clouds, don't worry, I'ma get it, granny, uh\nWhat happened? Now my daddy happy, mama called me up"
#seed = "I remember syrup sandwiches and crime allowances finesse a n5gga with some counterfeit"
print(sample_from_model(seed, 300, 1, predictorLength))

Uh, I got a halo
I level up every time God says so
Shooter on payroll
Lookin' like Pancho, lookin' like Pedro
Money tree for Tarzan
Ten dividends go grow when I say grow 
(more sure' s change I vote a try woof 
egg wanna Power my hit, she hope that get mistake forever 
On it is' em times, tired up Us to my deuce 
" Crosses turn my mad, who from using? 
Feel(cherish, yeah, go sensitive, yep(But I' m nothin with a Hell for die mistakes it 

Still 
We all you b5tch of women about to been on 
Who, key, way to and my But 
The through the vibe livin is kick him our thought' s boo, man) 
I got a'' water the denim is the one for? 
That gotta the Sippin, faggot, true 
Like nowadays and runnin to Kenny for like the cycle 

blog' € while a man bish 
In care, get be way hustler 

[Kendrick Lamar:] 
What ridin, huh stay" she loyal my Faded 
Crooked to day it' s do 
score the mean, the go- mirror, my scratch 
n5gga f5ckin' t slowed it, the Drake n5gga places! 
Every I want of real is you f5ck this a